[Back to Computer Networks guideline](Computer-Networks.html)

## **Congestion Control and Resource Sharing**

Chapter 5 ended with the sender-side bound

$$

\text{bytes in flight}\le \min(rwnd,cwnd).

$$

The receive window `rwnd` protects one receiver. The congestion window `cwnd` protects a shared path that the sender cannot inspect directly. This difference becomes important when several perfectly healthy endpoints send through the same 100 Mb/s router. Each receiver may advertise plenty of free memory, yet their combined traffic can arrive faster than the bottleneck link can transmit it. Packets then wait, latency grows, the finite queue eventually overflows, and retransmissions can add still more load.

Congestion control is therefore a **distributed resource-allocation problem**. End hosts choose sending rates from incomplete feedback, routers decide which queued packet to serve and when to signal pressure, and every flow changes the conditions observed by the others. A good design should use available capacity, keep delay and loss bounded, share resources according to an explicit policy, and remain stable while routes and traffic change.

::: {.callout-note}
On a first reading, follow one causal chain: excessive offered load creates a queue; a queue creates delay, loss, or an ECN mark; the sender interprets that signal and changes `cwnd` or its pacing rate. Queue scheduling decides **whose** packet leaves next, while traffic shaping decides **when** admitted traffic may enter.
:::

This chapter first builds that control loop, then connects classic TCP, modern congestion-control algorithms, fairness models, bufferbloat, schedulers, Active Queue Management (AQM), and Quality of Service (QoS). The Python examples are deliberately small and deterministic: they expose the mechanism rather than pretending to reproduce a production TCP stack or router.

### **What Is Network Congestion?**

#### **Offered Load, Capacity, and Queue Growth**

**Network congestion** occurs when traffic persistently demands more of a shared resource than that resource can serve within the required time. A checkout queue is a useful analogy. A short group of customers can wait while the cashier catches up; that is ordinary burst absorption. If customers continue arriving faster than the cashier serves them, the line cannot stabilize. A packet queue behaves the same way, except its waiting room is finite and excess arrivals are dropped or marked.

Let $A(t)$ be arrivals during one interval, $S(t)$ be the service made possible by the bottleneck, $B$ be the buffer limit, and $Q(t)$ be queued work. A simple queue recurrence is

$$

Q(t+1)=\min\left[B,\max\left(0,Q(t)+A(t)-S(t)\right)\right].

$$

If the long-run arrival rate $\lambda$ is below service rate $\mu$, idle periods can drain the queue. As utilization $\rho=\lambda/\mu$ approaches one, small fluctuations produce increasingly long waits. In an idealized M/M/1 queue, the mean system time is $1/(\mu-\lambda)$. Real Internet traffic is neither Poisson nor served by one isolated exponential server, so this is an intuition about the sharp approach to saturation, not a delay formula to apply blindly.

For a queue containing $Q$ bits before a link of rate $C$ bits/s, its immediate serialization backlog contributes approximately

$$

d_{queue}\approx \frac{Q}{C}.

$$

Thus a full 12.5 MB buffer before a 100 Mb/s link can contain about one second of waiting data even when the link reports 100% utilization.

![Three flows overload a finite bottleneck queue; persistent arrivals above service capacity create delay and overflow.](assets/congestion-queue-dynamics.svg){fig-alt="Three flows totaling 120 megabits per second enter a finite queue served by a 100 megabit per second link, with queue update equation and load response curves" width="98%"}

#### **Congestion vs Flow Control**

Flow control asks, **can this receiver accept more bytes?** Congestion control asks, **should the shared path carry more bytes now?** A sender can be flow-control limited when `rwnd < cwnd`, congestion-control limited when `cwnd < rwnd`, or application limited when it has too little data to fill either window. Increasing a socket buffer repairs only the first condition; it cannot create bottleneck capacity.

#### **Congestion Collapse**

Congestion is not merely "the link is busy." A busy link can be efficient. **Congestion collapse** means offered work rises while useful delivered work falls. This can happen when scarce downstream capacity is spent transmitting packets whose predecessors will time out, retransmissions duplicate data already in the network, packets consume several upstream hops before being discarded, or long queues make transport timers generate still more traffic. The network performs work, but less application data reaches its destination.

Modern congestion control tries to prevent that positive-feedback loop. A loss still consumes the capacity used on every hop before the drop, so "send until everything is lost" is not an efficient discovery strategy.

#### **Efficiency, Delay, Fairness, and Stability**

One number cannot describe congestion-control quality:

| Objective | Question | Failure mode when ignored |
|---|---|---|
| Efficiency | Is useful traffic close to available capacity? | an empty link despite queued application data |
| Delay | How long do packets wait, including tail latency? | bufferbloat and poor interactive response |
| Loss/mark cost | How much work is discarded or explicitly signaled? | retransmission waste or excessive control churn |
| Fairness | How is capacity divided among competing demands? | starvation, RTT bias, or policy violation |
| Stability | Do rates settle or oscillate violently? | synchronized bursts, repeated queue overflow |

The objectives can conflict. A tiny queue gives low delay but may drop a burst. A large queue absorbs a burst but can preserve a standing delay. Strict equality may be inappropriate when a voice call and a backup transfer have different service requirements. The design task is to make those trade-offs explicit.

In [1]:
from dataclasses import dataclass


@dataclass
class QueueResult:
    offered: int
    delivered: int
    dropped: int
    final_queue: int
    mean_queue_delay_steps: float


def simulate_constant_queue(offered_per_step, capacity=100, buffer=300, steps=30):
    """Simulate one finite FIFO queue in packet-sized units."""
    queue = 0
    delivered = 0
    dropped = 0
    delay_samples = []

    for _ in range(steps):
        # Arrivals first occupy free buffer space.
        admitted = min(offered_per_step, buffer - queue)
        dropped += offered_per_step - admitted
        queue += admitted

        # The bottleneck then serves at most `capacity` units.
        served = min(queue, capacity)
        queue -= served
        delivered += served

        # Backlog divided by service rate is a queue-delay estimate.
        delay_samples.append(queue / capacity)

    return QueueResult(
        offered=offered_per_step * steps,
        delivered=delivered,
        dropped=dropped,
        final_queue=queue,
        mean_queue_delay_steps=sum(delay_samples) / len(delay_samples),
    )


print("load/step | delivered | dropped | final queue | mean queue delay")
for load in (80, 100, 120):
    result = simulate_constant_queue(load)
    print(
        f"{load:9d} | {result.delivered:9d} | {result.dropped:7d} |"
        f" {result.final_queue:11d} | {result.mean_queue_delay_steps:7.2f} steps"
    )

load/step | delivered | dropped | final queue | mean queue delay
       80 |      2400 |       0 |           0 |    0.00 steps
      100 |      3000 |       0 |           0 |    0.00 steps
      120 |      3000 |     400 |         200 |    1.70 steps


The 80-unit source leaves spare capacity. The 100-unit source exactly matches service in this synchronized toy model. At 120 units, the queue reaches its limit and later packets are discarded even though useful throughput cannot exceed 100 units per step. The simulation does not add retransmissions; doing so without reducing the original source rate would make offered load still larger and demonstrates the route toward collapse.

### **Congestion Signals and Control Placement**

#### **Loss-Based, Delay-Based, and Explicit Signals**

A sender normally sees ACKs, timing, and transport feedback rather than the router's queue. It must convert those observations into a congestion estimate.

- **Loss-based control** treats a timeout, repeated ACK evidence, or another validated loss event as a sign that the path has been pushed too far. It is robust on ordinary Internet paths, but overflow is a late signal and not every loss is caused by congestion.
- **Delay-based control** compares current RTT with a minimum or baseline RTT. The difference approximates queueing, so the sender can slow before a drop. The difficult part is separating queue delay from route changes, delayed ACKs, receiver scheduling, and competition with aggressive loss-based flows.
- **Explicit control** lets the network mark or report congestion. The signal is earlier and less wasteful than dropping useful data, but endpoints and routers must support compatible semantics.

![Loss, increased RTT, and ECN are three different ways for an endpoint to learn about path pressure.](assets/congestion-signals.svg){fig-alt="Three sender router receiver timelines compare loss inferred through timeout or duplicate acknowledgments, delay inferred from RTT, and ECN feedback through CE ECE and CWR" width="98%"}

#### **End-to-End vs Network-Assisted Control**

The **end-to-end principle** favors intelligence at endpoints when the network cannot reliably maintain per-flow state. Classic TCP follows this approach: routers forward and may drop, while senders infer pressure. It deploys across heterogeneous networks, but an endpoint cannot distinguish every cause of loss or delay.

**Network-assisted control** supplies more information. A router may mark packets, schedule per class, enforce a rate, or reserve resources. This can improve precision and isolation inside an administrative domain, but it introduces configuration, trust, and compatibility requirements. The two approaches are not opposites: an ECN-capable TCP sender still runs an end-host control law, using a signal generated by a router.

#### **Implicit Feedback and Explicit Congestion Notification**

[RFC 3168](https://datatracker.ietf.org/doc/html/rfc3168) defines classic Explicit Congestion Notification (ECN). After negotiation, a sender marks packets as **ECN-Capable Transport (ECT)**. An AQM-enabled router experiencing incipient congestion may change the IP ECN field to **Congestion Experienced (CE)** instead of dropping the packet. For TCP, the receiver echoes this condition with **ECE**, and the sender reduces its congestion response and uses **CWR** to indicate that it reacted.

ECN does not reserve bandwidth, prove that every router participates, or eliminate all drops. Non-ECT traffic must still receive a congestion signal through dropping, and even ECN-capable traffic can be dropped because of corruption, hard overflow, policy, or severe congestion. The key benefit is that a router can communicate pressure without destroying the packet carrying useful application bytes.

In [2]:
def infer_congestion(base_rtt_ms, current_rtt_ms, sent_bytes,
                     lost_bytes=0, ce_marked_bytes=0):
    """Summarize the three observations available to an endpoint."""
    queue_delay = max(0.0, current_rtt_ms - base_rtt_ms)
    loss_fraction = lost_bytes / sent_bytes if sent_bytes else 0.0
    ce_fraction = ce_marked_bytes / sent_bytes if sent_bytes else 0.0

    if ce_fraction > 0:
        primary = "explicit ECN pressure"
    elif loss_fraction > 0:
        primary = "loss-based pressure"
    elif queue_delay > 0.25 * base_rtt_ms:
        primary = "delay-based early warning"
    else:
        primary = "no strong congestion signal"

    return queue_delay, loss_fraction, ce_fraction, primary


samples = [
    ("quiet path", 20, 22, 1_000_000, 0, 0),
    ("growing queue", 20, 38, 1_000_000, 0, 0),
    ("overflow loss", 20, 65, 1_000_000, 20_000, 0),
    ("ECN-enabled queue", 20, 34, 1_000_000, 0, 80_000),
]

for name, base, current, sent, lost, marked in samples:
    qd, loss, ce, interpretation = infer_congestion(
        base, current, sent, lost, marked
    )
    print(
        f"{name:18s}: queue={qd:4.0f} ms, loss={loss:5.1%}, "
        f"CE={ce:5.1%} -> {interpretation}"
    )

quiet path        : queue=   2 ms, loss= 0.0%, CE= 0.0% -> no strong congestion signal
growing queue     : queue=  18 ms, loss= 0.0%, CE= 0.0% -> delay-based early warning
overflow loss     : queue=  45 ms, loss= 2.0%, CE= 0.0% -> loss-based pressure
ECN-enabled queue : queue=  14 ms, loss= 0.0%, CE= 8.0% -> explicit ECN pressure


The classification thresholds above are educational, not protocol constants. A real controller filters samples, handles ACK aggregation and route changes, validates loss evidence, and changes its rate according to a carefully tested state machine.

### **Additive Increase and Multiplicative Decrease**

#### **Congestion Windows and Flight Size**

The congestion window limits unacknowledged data according to the sender's estimate of path capacity. Ignoring application and receiver limits,

$$

\text{flight size}\le cwnd,

$$

and a window-limited flow's approximate rate is

$$

x\approx \frac{cwnd}{RTT}.

$$

`cwnd` is permission, not a timer. ACK arrival releases previously occupied window space and creates an **ACK clock** that naturally relates sending to the rate at which the path delivers data. Modern stacks also use pacing to spread packets instead of releasing a complete window as one burst.

#### **AIMD Dynamics**

**Additive Increase, Multiplicative Decrease (AIMD)** probes for spare capacity slowly and backs away proportionally after congestion:

$$

w_{t+1}=\begin{cases}
w_t+\alpha, & \text{without congestion},\\
\beta w_t, & \text{after congestion},\quad 0<\beta<1.
\end{cases}

$$

For classic Reno congestion avoidance, the aggregate increase is roughly one Maximum Segment Size (MSS) per RTT and a loss response commonly halves the window. The asymmetry is intentional. Linear probing limits overshoot near capacity; proportional decrease makes a larger flow surrender more absolute capacity than a smaller flow.

```text
FOR each feedback round
    if a validated congestion event occurred
        window <- beta * window
    else
        window <- window + additive_step
    send no more than the current window and pace transmissions
```

#### **Convergence to Efficiency and Fairness**

For two idealized flows sharing capacity $C$, the line $x_1+x_2=C$ is efficient and $x_1=x_2$ is equally shared. Additive increase moves both rates in the same direction. A synchronized multiplicative decrease moves toward the origin, with the larger flow making the larger absolute reduction. Repeated cycles tend toward the intersection.

![AIMD trajectories approach the intersection of the capacity boundary and equal-rate line.](assets/aimd-fairness.svg){fig-alt="Two-flow rate plane with an efficiency line, an equal fairness line, and alternating additive increase and multiplicative decrease arrows converging toward their intersection" width="86%"}

The diagram explains a tendency, not a universal guarantee. Different RTTs change how often a flow increases, different algorithms use different signals and decrease factors, and flows may not share the same bottleneck. Equal throughput is also only one fairness policy.

A common descriptive metric is **Jain's fairness index**:

$$

J(x_1,\ldots,x_n)=\frac{(\sum_i x_i)^2}{n\sum_i x_i^2}.

$$

It ranges from $1/n$ when one of $n$ flows receives everything to 1 when all measured rates are equal. It does not say whether the total rate is efficient or whether unequal service was intentionally weighted.

In [3]:
def jain_index(rates):
    numerator = sum(rates) ** 2
    denominator = len(rates) * sum(rate ** 2 for rate in rates)
    return numerator / denominator if denominator else 0.0


def simulate_two_flow_aimd(rounds=28, capacity=60, alpha=1, beta=0.5):
    """Use a shared binary signal to expose AIMD's geometric tendency."""
    windows = [8.0, 36.0]
    trace = []

    for round_number in range(1, rounds + 1):
        congested = sum(windows) > capacity
        if congested:
            windows = [beta * window for window in windows]
            action = "multiplicative decrease"
        else:
            windows = [window + alpha for window in windows]
            action = "additive increase"

        trace.append((round_number, *windows, sum(windows), jain_index(windows), action))

    return trace


trace = simulate_two_flow_aimd()
print("round | flow A | flow B | total | Jain  | action")
for row in trace:
    round_number, a, b, total, fairness, action = row
    if round_number <= 5 or action.startswith("multiplicative") or round_number >= 25:
        print(
            f"{round_number:5d} | {a:6.1f} | {b:6.1f} |"
            f" {total:5.1f} | {fairness:5.3f} | {action}"
        )

round | flow A | flow B | total | Jain  | action
    1 |    9.0 |   37.0 |  46.0 | 0.730 | additive increase
    2 |   10.0 |   38.0 |  48.0 | 0.746 | additive increase
    3 |   11.0 |   39.0 |  50.0 | 0.761 | additive increase
    4 |   12.0 |   40.0 |  52.0 | 0.775 | additive increase
    5 |   13.0 |   41.0 |  54.0 | 0.788 | additive increase
   10 |    8.5 |   22.5 |  31.0 | 0.831 | multiplicative decrease
   25 |   23.5 |   37.5 |  61.0 | 0.950 | additive increase
   26 |   11.8 |   18.8 |  30.5 | 0.950 | multiplicative decrease
   27 |   12.8 |   19.8 |  32.5 | 0.956 | additive increase
   28 |   13.8 |   20.8 |  34.5 | 0.960 | additive increase


### **Classical TCP Congestion Control**

#### **Slow Start**

[RFC 5681](https://datatracker.ietf.org/doc/html/rfc5681) organizes classic TCP around `cwnd`, the slow-start threshold `ssthresh`, ACK feedback, and congestion events. A new or restarted connection does not know the path's available capacity. **Slow start** begins with an initial window and uses returning ACKs to expand rapidly. During slow start, each ACK of new data increases `cwnd` by at most one SMSS, so a fully utilized window grows by roughly one complete window per RTT: approximately 1, 2, 4, 8, and 16 segments.

The name is historical; its growth is exponential by round trip. [RFC 6928](https://datatracker.ietf.org/doc/html/rfc6928) experimentally permits an initial window up to

$$

IW=\min(10\,MSS,\max(2\,MSS,14600\text{ bytes})),

$$

commonly called IW10. A larger initial window reduces short-transfer latency but can create a larger startup burst, especially when pacing is absent.

![TCP grows rapidly below the slow-start threshold, then probes linearly in congestion avoidance and reduces after congestion.](assets/tcp-slow-start-congestion-avoidance.png){fig-alt="Congestion window chart showing exponential slow start, linear congestion avoidance, and decreases after loss" width="90%"}

*Figure source: [Fleshgrinder, TCP Slow-Start and Congestion Avoidance, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:TCP_Slow-Start_and_Congestion_Avoidance.svg), licensed under GPLv3.*

#### **Congestion Avoidance**

When `cwnd` reaches `ssthresh`, TCP enters **congestion avoidance**. A common per-ACK approximation is

$$

cwnd\leftarrow cwnd+\frac{SMSS^2}{cwnd},

$$

which accumulates to roughly one SMSS per RTT when a complete window is acknowledged. This is the additive part of AIMD. It searches more cautiously because the path has either approached a known limit or recently experienced congestion.

#### **Fast Retransmit and Fast Recovery**

Three duplicate ACKs are classic evidence that a segment is missing while later data continues to arrive. **Fast retransmit** repairs the gap before the retransmission timer expires. **Fast recovery** avoids returning all the way to one segment because the duplicate ACKs also prove that packets are leaving the network.

The classic distinction is:

- after a retransmission timeout, set $ssthresh=\max(FlightSize/2,2\,SMSS)$ and restart with a loss window of one SMSS;
- after three duplicate ACKs, reduce around half, retransmit the missing segment, and use duplicate ACKs to account for packets still draining from the path;
- after an ACK covering the recovery point, leave fast recovery near `ssthresh` and continue congestion avoidance.

Loss recovery and congestion response are related but not identical. SACK tells the sender **which bytes** are missing; the congestion-control state determines **how much data** the sender may inject while repairing them.

#### **Tahoe, Reno, and NewReno**

| Variant | Three-duplicate-ACK behavior | Multiple losses in one window | Main lesson |
|---|---|---|---|
| Tahoe | fast retransmit, then `cwnd` returns to one MSS | recovers conservatively | loss repair can be fast even when rate restart is slow |
| Reno | fast retransmit plus fast recovery | a partial ACK can end recovery too early | preserving ACK clock improves a single-loss case |
| NewReno | remains in recovery after a partial ACK | repairs another loss without waiting for an RTO | recovery-point state matters |
| SACK-capable TCP | reports received byte ranges | targets several gaps more precisely | richer receiver evidence complements the controller |

[RFC 6582](https://datatracker.ietf.org/doc/html/rfc6582) specifies NewReno's modification. Production stacks include many additional mechanisms, but this family remains the clearest foundation for interpreting a congestion-window trace.

```text
ON ACK of new data
    if cwnd < ssthresh:       cwnd <- cwnd + min(new_bytes, SMSS) # slow start
    else:                     increase about one MSS per RTT       # avoidance

ON three duplicate ACKs
    ssthresh <- max(FlightSize / 2, 2 MSS)
    fast retransmit the missing segment
    enter fast recovery

ON retransmission timeout
    ssthresh <- max(FlightSize / 2, 2 MSS)
    cwnd <- 1 MSS
    restart slow start
```

In [4]:
def simplified_reno(rounds=14, initial_cwnd=10.0, initial_ssthresh=32.0):
    """RTT-granularity Reno illustration, not a packet-level implementation."""
    cwnd = initial_cwnd
    ssthresh = initial_ssthresh
    events = {5: "three duplicate ACKs", 10: "timeout"}
    trace = []

    for rtt in range(1, rounds + 1):
        event = events.get(rtt, "ACK progress")

        if event == "three duplicate ACKs":
            ssthresh = max(cwnd / 2, 2.0)
            cwnd = ssthresh       # state after simplified fast recovery
            phase = "fast recovery -> avoidance"
        elif event == "timeout":
            ssthresh = max(cwnd / 2, 2.0)
            cwnd = 1.0            # restart probing from a loss window
            phase = "slow start restart"
        elif cwnd < ssthresh:
            cwnd = min(2 * cwnd, ssthresh)
            phase = "slow start"
        else:
            cwnd += 1.0
            phase = "congestion avoidance"

        trace.append((rtt, cwnd, ssthresh, event, phase))

    return trace


print("RTT | cwnd(MSS) | ssthresh | event                 | resulting phase")
for rtt, cwnd, threshold, event, phase in simplified_reno():
    print(f"{rtt:3d} | {cwnd:9.1f} | {threshold:8.1f} | {event:21s} | {phase}")

RTT | cwnd(MSS) | ssthresh | event                 | resulting phase
  1 |      20.0 |     32.0 | ACK progress          | slow start
  2 |      32.0 |     32.0 | ACK progress          | slow start
  3 |      33.0 |     32.0 | ACK progress          | congestion avoidance
  4 |      34.0 |     32.0 | ACK progress          | congestion avoidance
  5 |      17.0 |     17.0 | three duplicate ACKs  | fast recovery -> avoidance
  6 |      18.0 |     17.0 | ACK progress          | congestion avoidance
  7 |      19.0 |     17.0 | ACK progress          | congestion avoidance
  8 |      20.0 |     17.0 | ACK progress          | congestion avoidance
  9 |      21.0 |     17.0 | ACK progress          | congestion avoidance
 10 |       1.0 |     10.5 | timeout               | slow start restart
 11 |       2.0 |     10.5 | ACK progress          | slow start
 12 |       4.0 |     10.5 | ACK progress          | slow start
 13 |       8.0 |     10.5 | ACK progress          | slow start
 14 |      10.5

### **Modern Congestion-Control Algorithms**

Classic Reno uses a relatively simple window law and interprets loss as its principal congestion signal. Modern networks span high-bandwidth long-distance paths, shallow datacenter queues, wireless loss, paced hosts, and latency-sensitive traffic. New algorithms change either the growth function, the measured signal, or the path model.

![CUBIC, Vegas, BBR, and DCTCP use different measurements and control laws.](assets/modern-congestion-control.svg){fig-alt="Four panels compare CUBIC elapsed-time window growth, Vegas expected versus actual rate, BBR bandwidth and propagation RTT model, and DCTCP response to ECN-marked fraction" width="98%"}

#### **CUBIC**

[RFC 9438](https://datatracker.ietf.org/doc/html/rfc9438) standardizes **CUBIC**, the default TCP congestion controller in several major operating-system families. Its window is primarily a cubic function of elapsed time since a congestion event:

$$

W_{cubic}(t)=C(t-K)^3+W_{max},

$$

where $W_{max}$ is the window just before the previous reduction and

$$

K=\sqrt[3]{\frac{W_{max}(1-\beta)}{C}}.

$$

Immediately after loss, CUBIC grows quickly toward the known operating region, becomes cautious near $W_{max}$, and then probes more aggressively above it. The recommended decrease factor is $\beta=0.7$, so it retains more of the previous window than Reno's classic one-half reduction. Growth based mainly on elapsed time reduces, but does not erase, RTT-dependent competition. CUBIC also includes Reno-friendly behavior and safeguards omitted from this conceptual equation.

#### **Delay-Based Control and TCP Vegas**

**TCP Vegas** tries to detect queued data before overflow. It compares

$$

Expected=\frac{cwnd}{BaseRTT},\qquad Actual=\frac{cwnd}{RTT}.

$$

When current RTT rises while the window stays similar, actual delivery is lower than the empty-path estimate. A small difference suggests spare capacity; a large difference suggests a queue and causes Vegas to hold or reduce its window.

This can provide low delay among compatible flows. Coexistence is harder: a delay-sensitive flow may yield when a loss-based flow fills the queue, allowing the loss-based flow to claim more capacity. `BaseRTT` can also become stale after a route change or remain overestimated when a connection never observes an empty queue.

#### **Model-Based Control and BBR**

**BBR** estimates bottleneck bandwidth (`BtlBw`) from delivered-data samples and propagation RTT (`RTprop`) from low RTT samples. Their product estimates the bandwidth-delay product:

$$

BDP\approx BtlBw\times RTprop.

$$

The controller uses pacing and an in-flight bound to operate near that model, periodically probing because both bandwidth and path delay can change. The goal is to avoid making persistent overflow loss the only capacity signal. BBR is not "no congestion response" and does not guarantee zero queues or universal fairness. Version, queue policy, ACK behavior, and coexistence with other controllers materially affect outcomes. See the [Google Research BBR overview](https://research.google/pubs/bbr-congestion-based-congestion-control/) for the model's original motivation.

#### **Datacenter TCP and ECN**

**DCTCP** is designed for controlled datacenter environments with shallow ECN-marking queues. Instead of treating ECN as one binary event, it estimates the fraction $M$ of marked bytes:

$$

\alpha\leftarrow(1-g)\alpha+gM,

$$

and responds approximately as

$$

cwnd\leftarrow cwnd\left(1-\frac{\alpha}{2}\right).

$$

A small marked fraction causes a small reduction; widespread marking causes a larger one. This proportional response can keep queues shallow while preserving throughput in a configured fabric. [RFC 8257](https://datatracker.ietf.org/doc/html/rfc8257) explicitly treats DCTCP as a controlled-environment mechanism. Deploying it unchanged across the public Internet can create coexistence and safety problems.

#### **Algorithm Selection and Coexistence**

| Controller | Principal observation | Characteristic strength | Important caveat |
|---|---|---|---|
| Reno/NewReno | packet loss | simple, conservative baseline | underuses very large BDP paths after a reduction |
| CUBIC | loss plus elapsed time from prior maximum | scalable probing on high-speed paths | can still build a queue before loss/marking |
| Vegas-style | RTT above baseline | early queue avoidance | may yield to aggressive loss-based competitors |
| BBR-style | delivery rate and minimum RTT model | pacing around a path model | fairness and queue behavior depend on version and environment |
| DCTCP | fraction of ECN-marked bytes | fine-grained response in shallow datacenter queues | requires controlled marking and coexistence policy |

Selection is an experiment, not a label. Compare controllers under representative RTTs, bottleneck rates, queue disciplines, loss mechanisms, flow sizes, and competitors. A controller that wins one isolated bulk transfer may harm short-flow latency or coexistence.

In [5]:
from math import isclose


def cubic_window(t, w_max=100.0, beta=0.7, cubic_c=0.4):
    """Conceptual RFC 9438 cubic curve, with windows measured in MSS."""
    k = (w_max * (1 - beta) / cubic_c) ** (1 / 3)
    return cubic_c * (t - k) ** 3 + w_max


def dctcp_update(alpha, marked_fraction, cwnd, gain=1 / 16):
    """One DCTCP alpha update followed by proportional window reduction."""
    alpha = (1 - gain) * alpha + gain * marked_fraction
    return alpha, cwnd * (1 - alpha / 2)


print("CUBIC elapsed time after a loss (previous Wmax = 100 MSS)")
for seconds in (0, 1, 2, 3, 4, 5, 6):
    print(f"t={seconds}s -> W={cubic_window(seconds):6.1f} MSS")

print("\nDCTCP reaction to successive marked-byte fractions")
alpha, cwnd = 0.0, 100.0
for marked in (0.05, 0.10, 0.40, 0.00):
    alpha, cwnd = dctcp_update(alpha, marked, cwnd)
    print(f"M={marked:4.0%} -> alpha={alpha:6.3f}, cwnd={cwnd:6.1f} MSS")

# BBR's central path target is dimensional: bandwidth times propagation time.
btlbw_mbps = 200
rtprop_ms = 40
bdp_bytes = btlbw_mbps * 1_000_000 * (rtprop_ms / 1000) / 8
print(f"\n200 Mb/s x 40 ms -> BDP target = {bdp_bytes / 1_000_000:.2f} MB")

CUBIC elapsed time after a loss (previous Wmax = 100 MSS)
t=0s -> W=  70.0 MSS
t=1s -> W=  86.7 MSS
t=2s -> W=  95.6 MSS
t=3s -> W=  99.3 MSS
t=4s -> W= 100.0 MSS
t=5s -> W= 100.2 MSS
t=6s -> W= 102.3 MSS

DCTCP reaction to successive marked-byte fractions
M=  5% -> alpha= 0.003, cwnd=  99.8 MSS
M= 10% -> alpha= 0.009, cwnd=  99.4 MSS
M= 40% -> alpha= 0.034, cwnd=  97.7 MSS
M=  0% -> alpha= 0.032, cwnd=  96.2 MSS

200 Mb/s x 40 ms -> BDP target = 1.00 MB


### **Throughput and Fairness Models**

#### **Window, RTT, and Throughput**

A sender must keep approximately one bandwidth-delay product in flight to fill a path:

$$

BDP=C\times RTT.

$$

If `cwnd` is smaller than the BDP, the sender runs out of permitted data before ACKs return. If it is much larger, the excess cannot increase bottleneck service and instead waits in a queue. With MSS-sized segments,

$$

Throughput\lesssim \min\left(C,\frac{cwnd}{RTT},\frac{rwnd}{RTT},\text{application rate}\right).

$$

This formula separates common bottlenecks. A large `cwnd` cannot overcome a small `rwnd`; neither window helps an application that produces data slowly.

#### **TCP Throughput Under Loss**

For a long-lived Reno-like flow in congestion avoidance with independent random loss probability $p$, a classic square-root approximation is

$$

T\approx K\frac{MSS}{RTT\sqrt{p}},

$$

where $K$ depends on ACK and loss assumptions. It explains three tendencies: larger segments carry more useful data per window unit, longer RTT slows additive recovery, and even small loss rates can substantially reduce throughput.

It is not a universal TCP speed formula. Timeouts, correlated burst loss, short flows, delayed ACKs, receiver limits, CUBIC, BBR, pacing, offload, and application behavior violate its assumptions. Use it to reason about direction and scale, then measure the actual controller.

#### **RTT Fairness and Flow Competition**

An ACK-clocked additive increase happens per feedback round. A short-RTT flow receives more feedback rounds per second and may increase faster than a long-RTT flow at the same bottleneck. Opening several parallel flows can also claim several shares of a per-flow scheduler or AIMD competition. "TCP-friendly" therefore depends on which TCP, RTT distribution, number of flows, and queue behavior.

#### **Max-Min and Proportional Fairness**

**Max-min fairness** increases every unsatisfied flow together until a bottleneck saturates; a flow can receive more only if doing so does not reduce another flow with an equal or smaller allocation. The water-filling analogy is literal: flows with small demands finish first, and the remaining capacity is shared among the rest.

**Proportional fairness** chooses feasible rates that maximize

$$

\sum_i \log x_i.

$$

This balances total efficiency with diminishing benefit from giving still more capacity to an already fast flow. Weighted variants use $\sum_i w_i\log x_i$. Neither policy appears automatically because a TCP algorithm is installed; the combined endpoint control laws, scheduler, route constraints, and administrative policy determine the observed allocation.

In [6]:
from math import sqrt


def reno_square_root_mbps(rtt_ms, loss_probability, mss_bytes=1460):
    """Directional Reno throughput estimate under restrictive assumptions."""
    k = sqrt(3 / 2)  # one common approximation constant
    bytes_per_second = k * mss_bytes / ((rtt_ms / 1000) * sqrt(loss_probability))
    return bytes_per_second * 8 / 1_000_000


print("RTT  | loss     | approximate Reno-like throughput")
for rtt_ms, probability in ((20, 1e-4), (80, 1e-4), (20, 1e-3), (80, 1e-3)):
    rate = reno_square_root_mbps(rtt_ms, probability)
    print(f"{rtt_ms:3d}ms | {probability:8.4%} | {rate:8.2f} Mb/s")

RTT  | loss     | approximate Reno-like throughput
 20ms |  0.0100% |    71.53 Mb/s
 80ms |  0.0100% |    17.88 Mb/s
 20ms |  0.1000% |    22.62 Mb/s
 80ms |  0.1000% |     5.65 Mb/s


In [7]:
def max_min_single_link(capacity, demands):
    """Water-fill one bottleneck while respecting each flow's demand cap."""
    allocation = {name: 0.0 for name in demands}
    active = set(demands)
    remaining = float(capacity)

    while active:
        equal_share = remaining / len(active)
        satisfied = [
            name for name in active
            if demands[name] - allocation[name] <= equal_share
        ]

        if not satisfied:
            for name in active:
                allocation[name] += equal_share
            break

        for name in satisfied:
            needed = demands[name] - allocation[name]
            allocation[name] += needed
            remaining -= needed
            active.remove(name)

    return allocation


demands = {"voice": 10, "video": 40, "backup": 80}
allocation = max_min_single_link(capacity=90, demands=demands)
print("Max-min allocation on one 90 Mb/s bottleneck")
for flow, rate in allocation.items():
    print(f"{flow:7s}: demand={demands[flow]:2d}, allocated={rate:4.1f} Mb/s")
print(f"used={sum(allocation.values()):.1f} Mb/s")

Max-min allocation on one 90 Mb/s bottleneck
voice  : demand=10, allocated=10.0 Mb/s
video  : demand=40, allocated=40.0 Mb/s
backup : demand=80, allocated=40.0 Mb/s
used=90.0 Mb/s


The low-demand voice flow receives everything it needs. The remaining 80 Mb/s is split equally between the two still-unsatisfied flows. Their unequal unmet demand does not let the backup flow reduce the video's smaller allocation.

### **Queues, Buffers, and Bufferbloat**

#### **Drop-Tail Queues**

A **drop-tail** queue admits packets while space remains and drops an arrival when the hard limit is reached. It is simple, work-conserving, and independent of transport details. Its signal arrives late: the queue is already full. Several flows can then observe losses together, reduce together, leave the link temporarily underused, and grow together again. Bursty flows can also lose several adjacent packets.

#### **Buffer Sizing**

A buffer should absorb expected bursts and scheduler variation without becoming a permanent warehouse. The path BDP is a useful scale because it measures one round trip of work, but "one BDP of buffering" is not a universal rule. The appropriate limit changes with the number of flows, pacing, controller behavior, target latency, access-link rate, scheduler, and AQM. Statistical multiplexing can reduce the buffer needed to keep a busy link utilized, while a small number of synchronized or bursty flows may need different treatment.

#### **Standing Queues and Latency Inflation**

**Bufferbloat** occurs when oversized or poorly managed buffers remain occupied and add substantial latency without increasing bottleneck throughput. A speed test can report the full access rate while a voice call, game, DNS lookup, or remote shell waits behind seconds of bulk-transfer data. This is why unloaded ping time and loaded latency answer different questions.

![The same-capacity link can absorb a burst and drain, or preserve a standing queue that adds latency without adding throughput.](assets/bufferbloat.svg){fig-alt="Two bottleneck queues compare a short queue spike that drains with an oversized standing queue that maintains high delay at the same link capacity" width="96%"}

#### **Burst Absorption vs Persistent Delay**

A short queue is not automatically bad. If a 1 Gb/s source briefly sends into a 100 Mb/s link, some buffering allows the burst to complete without immediate loss. The distinction is whether the backlog drains. A **transient queue** represents useful smoothing; a **standing queue** means offered load and control behavior keep replacing every departing packet.

Operationally, compare queue delay with the application's latency budget, not only with buffer occupancy. One megabyte is about 8 ms on a 1 Gb/s link, 80 ms on a 100 Mb/s link, and 800 ms on a 10 Mb/s link.

In [8]:
def simulate_burst(buffer_size, capacity=100, steps=35):
    """Compare a short burst under two queue limits."""
    queue = delivered = dropped = 0
    queue_delays = []

    for step in range(steps):
        # Baseline, a four-step burst, then traffic just below link capacity.
        arrivals = 80 if step < 5 else (220 if step < 9 else 95)
        admitted = min(arrivals, buffer_size - queue)
        dropped += arrivals - admitted
        queue += admitted

        served = min(queue, capacity)
        queue -= served
        delivered += served
        queue_delays.append(queue / capacity)

    ordered = sorted(queue_delays)
    p95 = ordered[int(0.95 * (len(ordered) - 1))]
    return delivered, dropped, queue, sum(queue_delays) / steps, p95


print("buffer | delivered | dropped | end queue | mean delay | p95 delay")
for size in (200, 1000):
    delivered, dropped, queue, mean_delay, p95 = simulate_burst(size)
    print(
        f"{size:6d} | {delivered:9d} | {dropped:7d} | {queue:9d} |"
        f" {mean_delay:8.2f} | {p95:8.2f} steps"
    )

buffer | delivered | dropped | end queue | mean delay | p95 delay
   200 |      3370 |     380 |         0 |     0.39 |     1.00 steps
  1000 |      3400 |       0 |       350 |     3.41 |     4.70 steps


The larger buffer discards less of the burst, but it preserves a much larger backlog after the burst ends. Whether that trade is acceptable depends on delay requirements and whether endpoint control or AQM makes the queue drain. Buffer size alone is not a congestion-control algorithm.

### **Queue Scheduling**

#### **FIFO and Priority Queuing**

A queue can contain packets from many flows and service classes. **FIFO** sends them in arrival order. It is simple, but a large burst from one flow can delay an unrelated short packet. **Strict priority** always serves the highest non-empty class. This protects urgent traffic only while that class is controlled; an unlimited high-priority source can starve every lower queue.

![FIFO, strict priority, and DRR differ in isolation, starvation risk, and treatment of variable packet sizes.](assets/scheduler-comparison.svg){fig-alt="Three panels show a single FIFO queue, high medium and low strict-priority queues, and rotating deficit-round-robin per-flow queues with byte credits" width="98%"}

#### **Round Robin and Deficit Round Robin**

Plain **Round Robin (RR)** visits each non-empty queue and sends one packet. One packet is not one equal amount of service: a flow of 1500-byte packets receives more bytes than a flow of 300-byte packets.

**Deficit Round Robin (DRR)** measures service in bytes. Queue $i$ receives a quantum $q_i$ on each visit and keeps a deficit counter $D_i$:

```text
REPEAT while any queue is non-empty
    FOR each active queue i in cyclic order
        D[i] <- D[i] + quantum[i]
        WHILE queue i is non-empty AND head_size(i) <= D[i]
            transmit head packet
            D[i] <- D[i] - packet_size
        if queue i becomes empty
            D[i] <- 0
```

Unused credit carries forward, so a packet larger than one quantum eventually becomes eligible. Larger quanta implement weighted service. DRR is practical because it handles variable packet sizes without calculating an exact virtual finish time for every packet.

#### **Fair Queuing and Weighted Fair Queuing**

Ideal **Generalized Processor Sharing (GPS)** would serve infinitesimal pieces of every backlogged flow simultaneously. Packet links cannot do that. **Fair Queuing (FQ)** approximates GPS by assigning packets virtual finish times; **Weighted Fair Queuing (WFQ)** gives flow $i$ a configured weight $w_i$, so continuously backlogged flows tend toward shares proportional to their weights.

![Weighted Fair Queuing separates traffic into queues and schedules weighted service onto one output link.](assets/weighted-fair-queuing.png){fig-alt="Weighted fair queuing diagram with multiple classified queues feeding a weighted scheduler and one output" width="86%"}

*Figure source: [Lorenzo David and Luca Ghio, QoS weighted fair queuing, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:QoS_weighted_fair_queuing.svg), licensed under CC BY-SA 4.0.*

WFQ improves isolation, but the result depends on classification. Per-five-tuple queues can be manipulated by opening many flows; per-subscriber queues can protect households but combine unrelated applications. Encryption can also hide application labels, making endpoint, class, or aggregate policies more practical than deep packet inspection.

#### **Scheduling Isolation and Starvation**

| Scheduler | State | Strength | Risk |
|---|---|---|---|
| FIFO | one queue | minimal cost, preserves arrival order | head-of-line delay and no flow isolation |
| strict priority | one queue per class | bounded delay for controlled high-priority traffic | lower classes can starve |
| RR | per-flow/class queues | simple rotation | unequal bytes when packet sizes differ |
| DRR | queues plus byte deficits | efficient weighted byte fairness | burstiness depends on quantum and classification |
| FQ/WFQ | queues plus virtual service state | close approximation to fair fluid sharing | more state and classification complexity |

Scheduling chooses the next packet among those already admitted. It does not by itself decide whether the total queue is too long; that is the role of queue limits and AQM.

In [9]:
from collections import deque


def deficit_round_robin(packet_queues, quanta):
    """Return a DRR transmission trace for variable-size packets."""
    queues = {name: deque(packets) for name, packets in packet_queues.items()}
    deficit = {name: 0 for name in queues}
    transmitted = []

    while any(queues.values()):
        for name, queue in queues.items():
            if not queue:
                deficit[name] = 0
                continue

            deficit[name] += quanta[name]
            while queue and queue[0] <= deficit[name]:
                size = queue.popleft()
                deficit[name] -= size
                transmitted.append((name, size, deficit[name]))

    return transmitted


packets = {
    "A": [900, 900, 900],
    "B": [1500, 1500],
    "C": [500, 500, 500, 500],
}
quanta = {"A": 1000, "B": 1500, "C": 500}
trace = deficit_round_robin(packets, quanta)

served = {name: 0 for name in packets}
print("order | flow | packet bytes | remaining credit")
for order, (name, size, credit) in enumerate(trace, start=1):
    served[name] += size
    print(f"{order:5d} | {name:4s} | {size:12d} | {credit:16d}")
print("served bytes:", served)

order | flow | packet bytes | remaining credit
    1 | A    |          900 |              100
    2 | B    |         1500 |                0
    3 | C    |          500 |                0
    4 | A    |          900 |              200
    5 | B    |         1500 |                0
    6 | C    |          500 |                0
    7 | A    |          900 |              300
    8 | C    |          500 |                0
    9 | C    |          500 |                0
served bytes: {'A': 2700, 'B': 3000, 'C': 2000}


### **Active Queue Management**

An AQM algorithm observes a queue and deliberately marks or drops selected packets **before unavoidable hard overflow**. The early signal gives responsive senders time to reduce their offered load. AQM is not extra link capacity, and it cannot control a sender that ignores all signals; policing may still be required at a trust boundary.

#### **Random Early Detection**

**Random Early Detection (RED)** maintains an exponentially weighted average queue length. Below a minimum threshold it admits normally. Between minimum and maximum thresholds it raises a probabilistic mark/drop rate. Above the maximum it signals aggressively. Randomization tries to avoid every flow reacting to the same deterministic queue limit.

![RED increases its random mark or drop probability as average queue length moves between configured thresholds.](assets/red-algorithm.png){fig-alt="Random Early Detection flowchart using average queue length, minimum and maximum thresholds, and a probability-based packet decision" width="72%"}

*Figure source: [helix84, Random Early Detection algorithm, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Random_Early_Detection_algorithm_en.svg), licensed under CC BY-SA 3.0.*

RED can distinguish a short instantaneous burst from persistent occupancy through averaging, but threshold, weight, and probability tuning are sensitive to rates, RTTs, and traffic mixes. Poor configuration can signal too late, underuse the link, or behave little better than drop-tail.

#### **CoDel and PIE**

**Controlled Delay (CoDel)** measures each packet's **sojourn time** from queue entry to departure. If the minimum sojourn time remains above a target for an interval, the queue is persistently bloated rather than experiencing one harmless burst. CoDel enters a controlled marking/dropping state whose event spacing becomes more aggressive until delay improves. [RFC 8289](https://datatracker.ietf.org/doc/html/rfc8289) gives 5 ms target and 100 ms interval as defaults for normal Internet paths, while noting that unusual environments may require different values.

**PIE** estimates current queueing delay and applies a proportional-integral-style controller to its mark/drop probability. Error from a delay target and the direction of delay change both matter. Unlike CoDel's per-packet sojourn measurement, PIE can operate from queue-length and departure-rate estimates without timestamping each packet. [RFC 8033](https://datatracker.ietf.org/doc/html/rfc8033) specifies the algorithm.

![Drop-tail waits for overflow, while RED, CoDel, and PIE create earlier signals using different queue observations.](assets/aqm-comparison.svg){fig-alt="Four queue timelines compare hard-limit drop-tail, average-queue RED, sojourn-time CoDel, and feedback-controlled PIE" width="98%"}

**FQ-CoDel** combines flow queueing with a CoDel instance so sparse flows can bypass a bulk flow's backlog while persistently bloated queues still receive AQM signals. [RFC 8290](https://datatracker.ietf.org/doc/html/rfc8290) describes this composition. FQ supplies isolation; CoDel controls standing delay. The two roles are complementary.

#### **AQM with Explicit Congestion Notification**

For an ECT packet, an AQM decision can set CE instead of discarding data. The same queue policy therefore becomes a lower-cost signal when both endpoints support ECN. A router must still drop non-ECT packets when it needs to signal them, and all traffic remains subject to hard limits.

[RFC 7567](https://datatracker.ietf.org/doc/html/rfc7567) recommends active queue management as a way to control queueing delay and avoid relying only on overflow. Deployment requires measurement: a target appropriate for a broadband access queue may not fit a datacenter switch or satellite path.

In [10]:
def red_probability(avg_queue, minimum=20, maximum=80, max_probability=0.10):
    """Linear RED probability region used for explanation."""
    if avg_queue < minimum:
        return 0.0
    if avg_queue >= maximum:
        return 1.0
    fraction = (avg_queue - minimum) / (maximum - minimum)
    return max_probability * fraction


def codel_persistent_delay(sojourn_ms, target_ms=5):
    """One teaching interval: persistent only if its minimum stays high."""
    return min(sojourn_ms) > target_ms


def simplified_pie_update(probability, delay_ms, previous_delay_ms,
                          target_ms=15, alpha=0.004, beta=0.002):
    """Pedagogical feedback update, not the RFC's complete implementation."""
    error = delay_ms - target_ms
    trend = delay_ms - previous_delay_ms
    return min(1.0, max(0.0, probability + alpha * error + beta * trend))


print("RED average queue -> mark/drop probability")
for queue in (10, 20, 35, 50, 65, 80):
    print(f"{queue:2d} packets -> {red_probability(queue):6.2%}")

print("\nCoDel teaching intervals")
for samples in ([2, 8, 11, 4], [8, 9, 12, 10]):
    print(f"{samples} ms -> persistent={codel_persistent_delay(samples)}")

print("\nSimplified PIE probability updates")
p, previous = 0.02, 12
for delay in (18, 24, 20, 14):
    p = simplified_pie_update(p, delay, previous)
    print(f"delay={delay:2d} ms -> probability={p:5.1%}")
    previous = delay

RED average queue -> mark/drop probability
10 packets ->  0.00%
20 packets ->  0.00%
35 packets ->  2.50%
50 packets ->  5.00%
65 packets ->  7.50%
80 packets -> 100.00%

CoDel teaching intervals
[2, 8, 11, 4] ms -> persistent=False
[8, 9, 12, 10] ms -> persistent=True

Simplified PIE probability updates
delay=18 ms -> probability= 4.4%
delay=24 ms -> probability= 9.2%
delay=20 ms -> probability=10.4%
delay=14 ms -> probability= 8.8%


The code captures each algorithm's control variable, not production timing, randomization, burst allowance, ECN behavior, or safeguards. A router implementation should follow the relevant specification and be tested under realistic traffic.

### **Traffic Shaping and Quality of Service**

#### **Leaky Bucket and Token Bucket**

Congestion control adapts to observed path conditions. **Traffic shaping** enforces a traffic profile at an edge, often before packets reach the congested resource.

A **leaky bucket** emits at a bounded drain rate, smoothing bursts toward a regular output. A **token bucket** adds tokens at rate $r$ up to capacity $B$. Sending a packet of size $L$ consumes $L$ tokens. The update is

$$

tokens(t+\Delta t)=\min(B,tokens(t)+r\Delta t),

$$

and a packet can pass immediately when `tokens >= L`. Over an interval $T$, the profile permits at most approximately

$$

B+rT

$$

bytes: $B$ allows a burst, while $r$ limits the long-run average.

![A token bucket accumulates credits at a fixed rate and spends them to admit a bounded burst.](assets/token-bucket.png){fig-alt="Token bucket diagram where incoming traffic consumes generated tokens before conforming traffic leaves" width="82%"}

*Figure source: [Lorenzo David and Luca Ghio, QoS token bucket, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:QoS_tocken_bucket.svg), licensed under CC BY-SA 4.0.*

#### **Policing vs Shaping**

A **policer** immediately forwards conforming traffic and drops or remarks nonconforming traffic. It protects a boundary without storing a potentially large backlog. A **shaper** delays nonconforming traffic until the profile permits it, trading memory and latency for fewer drops. Neither creates bandwidth; both make offered traffic match a contract.

| Mechanism | Excess packet | Typical placement | Main trade-off |
|---|---|---|---|
| policer | drop or lower its class | ingress/trust boundary | no shaping delay, but loss can be bursty |
| shaper | queue until tokens become available | egress/customer edge | smooth output, but adds delay and needs memory |
| congestion controller | adapt future sending | transport endpoint | responds to path feedback rather than a fixed contract |

#### **Differentiated Services and Service Classes**

The Differentiated Services architecture in [RFC 2475](https://datatracker.ietf.org/doc/html/rfc2475) places a DS codepoint in the IP header and assigns **per-hop behaviors** inside a domain. Routers can map classes to queues, weights, drop precedence, or rate limits without maintaining reservation state for every end-to-end flow.

A DSCP is not an end-to-end guarantee. Domains may remark it, tunnel it, ignore it, or map it differently. A trustworthy design classifies and polices traffic at an edge, provisions each class, and monitors whether the configured scheduler actually meets latency and loss objectives. Marking every packet "high priority" only destroys the distinction.

#### **Admission Control and Reserved Resources**

When an application requires a hard service bound, rate adaptation alone may be insufficient. **Admission control** accepts a new demand only if the system can still honor existing commitments. Reservation can allocate rate, buffer, scheduler weight, or a time slot. This is practical inside controlled WANs, datacenters, or real-time systems, but difficult across independently administered public networks.

Admission control and DiffServ answer policy questions; congestion control still handles variation and failure. A reserved service without enforcement can be oversubscribed, while perfectly responsive endpoints cannot invent capacity after too many guaranteed flows have been admitted.

In [11]:
def token_bucket_shape(arrivals, rate_bytes_per_second, bucket_bytes):
    """Shape (arrival_time, packet_size) records and return departure times."""
    tokens = float(bucket_bytes)  # begin with a full bucket: one burst is allowed
    now = 0.0
    last_update = 0.0
    departures = []

    for arrival_time, packet_size in arrivals:
        # A queued earlier packet can keep `now` ahead of a later arrival.
        now = max(now, arrival_time)
        tokens = min(
            bucket_bytes,
            tokens + rate_bytes_per_second * (now - last_update),
        )
        last_update = now

        if tokens < packet_size:
            # Wait only long enough to generate the missing credits.
            wait = (packet_size - tokens) / rate_bytes_per_second
            now += wait
            tokens = 0.0
            last_update = now
        else:
            tokens -= packet_size

        departures.append((arrival_time, now, packet_size, now - arrival_time))

    return departures


# Five packets arrive as one burst; another arrives after one second.
traffic = [(0.0, 1000)] * 5 + [(1.0, 1000)]
trace = token_bucket_shape(traffic, rate_bytes_per_second=1000, bucket_bytes=3000)

print("arrival | departure | bytes | shaping delay")
for arrival, departure, size, delay in trace:
    print(f"{arrival:7.1f} | {departure:9.1f} | {size:5d} | {delay:13.1f} s")

arrival | departure | bytes | shaping delay
    0.0 |       0.0 |  1000 |           0.0 s
    0.0 |       0.0 |  1000 |           0.0 s
    0.0 |       0.0 |  1000 |           0.0 s
    0.0 |       1.0 |  1000 |           1.0 s
    0.0 |       2.0 |  1000 |           2.0 s
    1.0 |       3.0 |  1000 |           2.0 s


The full 3000-byte bucket releases three packets immediately. Later packets wait for credits at 1000 bytes/s. The bucket permits a controlled burst without changing the long-run rate; a pure constant-rate leaky bucket would smooth even the first three departures.

### **Simulating Congestion-Window Evolution**

The next experiment places three control laws under the same **stylized** congestion-event schedule. It is not a claim that Reno, CUBIC, and DCTCP would see identical events on a real path. Their signals change the queue, which changes future signals. Holding the event schedule fixed is useful here because it isolates the window-update rule.

- Reno adds one MSS per round and halves on a congestion event.
- CUBIC records the prior maximum, keeps 70%, and follows its cubic epoch between events.
- DCTCP receives a 30% marked-byte fraction at each event, smooths `alpha`, and reduces proportionally.

The comparison should be read as a state-machine trace, not a benchmark.

In [12]:
def compare_window_controllers(rounds=26, event_rounds=(8, 16, 24)):
    reno = 20.0
    cubic = 20.0
    cubic_wmax = 20.0
    cubic_epoch = 0
    dctcp = 20.0
    alpha = 0.0
    trace = []

    for round_number in range(1, rounds + 1):
        event = round_number in event_rounds

        # Reno AIMD.
        reno = reno * 0.5 if event else reno + 1

        # Simplified CUBIC epoch using the conceptual curve from RFC 9438.
        if event:
            cubic_wmax = cubic
            cubic *= 0.7
            cubic_epoch = 0
        else:
            cubic_epoch += 1
            k = (cubic_wmax * 0.3 / 0.4) ** (1 / 3)
            curve = 0.4 * (cubic_epoch - k) ** 3 + cubic_wmax
            cubic = max(cubic, curve)

        # DCTCP sees a fraction, not merely a binary event.
        marked_fraction = 0.30 if event else 0.0
        alpha = (15 / 16) * alpha + (1 / 16) * marked_fraction
        dctcp = dctcp * (1 - alpha / 2) if event else dctcp + 1

        trace.append((round_number, event, reno, cubic, dctcp, alpha))

    return trace


print("round | signal | Reno | CUBIC | DCTCP | DCTCP alpha")
for row in compare_window_controllers():
    round_number, event, reno, cubic, dctcp, alpha = row
    if round_number <= 3 or event or round_number in (9, 17, 25, 26):
        print(
            f"{round_number:5d} | {str(event):6s} | {reno:4.1f} |"
            f" {cubic:5.1f} | {dctcp:6.1f} | {alpha:11.3f}"
        )

round | signal | Reno | CUBIC | DCTCP | DCTCP alpha
    1 | False  | 21.0 |  20.0 |   21.0 |       0.000
    2 | False  | 22.0 |  20.0 |   22.0 |       0.000
    3 | False  | 23.0 |  20.1 |   23.0 |       0.000
    8 | True   | 13.5 |  40.1 |   26.7 |       0.019
    9 | False  | 14.5 |  51.0 |   27.7 |       0.018
   16 | True   | 10.2 |  52.1 |   33.2 |       0.030
   17 | False  | 11.2 |  65.4 |   34.2 |       0.028
   24 | True   |  8.6 |  61.1 |   39.5 |       0.037
   25 | False  |  9.6 |  76.1 |   40.5 |       0.034
   26 | False  | 10.6 |  83.9 |   41.5 |       0.032


The binary Reno event produces the sharpest decrease. CUBIC's retained window and epoch shape return it toward its prior operating region. DCTCP's initial response is mild because only part of the traffic was marked and `alpha` is smoothed; repeated marking would accumulate into a stronger reduction. Production implementations include bounds, pacing, idle restart, recovery interaction, and many details intentionally absent here.

### **Comparing Throughput, Delay, and Fairness**

A congestion-control experiment should report the outcomes users and competing flows experience, not only one transfer's average throughput.

| Scenario | Primary concern | Mechanisms worth testing | Metrics that reveal failure |
|---|---|---|---|
| home access link during upload | loaded latency | pacing, AQM, FQ-CoDel, sensible buffer | p50/p95 RTT under load, goodput, drop/CE rate |
| long-RTT bulk replication | filling a large BDP | CUBIC or model-based control, adequate windows | steady goodput, recovery time, queue occupancy |
| datacenter fan-in | shallow queues and synchronized bursts | ECN, DCTCP-style response, per-class limits | tail flow-completion time, marking fraction, incast loss |
| multi-tenant uplink | isolation | FQ/WFQ/DRR, policing, weighted classes | per-tenant rate, Jain index, starvation |
| voice beside backup traffic | latency plus minimum useful rate | classification, shaping, bounded priority | jitter, deadline misses, lower-class progress |

#### **A Reproducible Evaluation Pipeline**

1. State the topology, bottleneck capacity, base RTT, buffer/AQM, scheduler, controller, MSS, and number of flows.
2. Separate **offered rate**, **wire throughput**, and **application goodput**. Retransmitted bytes are throughput but not new goodput.
3. Measure unloaded and loaded RTT distributions, not only averages.
4. Record drops, ECN marks, retransmissions, queue occupancy or sojourn time, and controller state where available.
5. Vary flow start times, RTTs, flow sizes, reverse-path traffic, and competitors. One synchronized run can hide instability.
6. Report efficiency and allocation together. Jain index without total utilization can call two equally starved flows "perfectly fair."

The compact evaluator below demonstrates this distinction with synthetic snapshots.

In [13]:
def percentile(values, fraction):
    ordered = sorted(values)
    index = round((len(ordered) - 1) * fraction)
    return ordered[index]


def summarize_experiment(name, capacity_mbps, flow_rates, rtt_samples_ms,
                         retransmitted_mbps=0.0, ce_fraction=0.0):
    goodput = sum(flow_rates)
    wire_rate = goodput + retransmitted_mbps
    return {
        "name": name,
        "efficiency": goodput / capacity_mbps,
        "wire_load": wire_rate / capacity_mbps,
        "fairness": jain_index(flow_rates),
        "p95_rtt": percentile(rtt_samples_ms, 0.95),
        "ce": ce_fraction,
    }


experiments = [
    summarize_experiment(
        "drop-tail", 100, [48, 46], [20, 25, 60, 180, 240],
        retransmitted_mbps=8
    ),
    summarize_experiment(
        "AQM + FQ", 100, [47, 47], [20, 22, 24, 28, 32],
        retransmitted_mbps=1, ce_fraction=0.06
    ),
    summarize_experiment(
        "unfair", 100, [85, 9], [20, 25, 35, 70, 90],
        retransmitted_mbps=2
    ),
]

print("experiment | goodput efficiency | wire load | Jain | p95 RTT | CE")
for item in experiments:
    print(
        f"{item['name']:10s} | {item['efficiency']:17.1%} |"
        f" {item['wire_load']:9.1%} | {item['fairness']:5.3f} |"
        f" {item['p95_rtt']:7.0f} ms | {item['ce']:4.1%}"
    )

experiment | goodput efficiency | wire load | Jain | p95 RTT | CE
drop-tail  |             94.0% |    102.0% | 1.000 |     240 ms | 0.0%
AQM + FQ   |             94.0% |     95.0% | 1.000 |      32 ms | 6.0%
unfair     |             94.0% |     96.0% | 0.605 |      90 ms | 0.0%


The first two snapshots have the same useful throughput and equal sharing, but the drop-tail case sends more retransmitted traffic and has much worse tail latency. The third keeps total goodput high while one flow dominates. No single metric would expose all three differences.

### **Observing and Troubleshooting Congestion**

At an endpoint, correlate transport state with packet evidence and a loaded-path test. On Windows:

```powershell
Get-NetTCPConnection
netstat -s
netsh interface tcp show global
Get-Counter '\TCPv4\Segments Retransmitted/sec'
```

On a Linux router or endpoint, useful views include:

```bash
ss -ti
tc -s qdisc show
ip -s link
```

`ss -ti` can expose controller, RTT, pacing, congestion window, and delivery-rate information when the kernel provides it. `tc -s qdisc` shows the active scheduler/AQM and counters; interpreting endpoint behavior without checking the bottleneck queue can miss the actual policy.

Wireshark filters include:

```text
tcp.analysis.retransmission
tcp.analysis.duplicate_ack
tcp.analysis.lost_segment
tcp.flags.ece == 1
tcp.flags.cwr == 1
ip.dsfield.ecn == 3
```

Packet capture cannot directly reveal every internal `cwnd` update, and capture offload can distort segment sizes. Compare both endpoints when possible, identify the real bottleneck, measure baseline RTT before load, and avoid concluding that every loss is congestion merely because TCP reduced its rate.

### **Summary**

- Congestion begins when persistent offered load exceeds a shared service rate; queueing, loss, and wasted work are consequences, not definitions.
- Flow control protects receiver memory, while congestion control protects the path and competing traffic through `cwnd`, pacing, and feedback.
- Loss, delay, and ECN expose different evidence. A controller's behavior depends on both its signal and its window or rate-update law.
- AIMD explains the classic probe-and-back-off pattern. Slow start finds an initial operating region; congestion avoidance, fast recovery, and timeout handling respond at different timescales.
- CUBIC changes the growth function, Vegas observes delay, BBR models bandwidth and propagation RTT, and DCTCP responds proportionally to ECN marking in controlled fabrics.
- Throughput must be interpreted with RTT, loss, BDP, application demand, and receiver limits. Fairness requires an explicit allocation definition and cannot replace efficiency or latency measurement.
- Buffer size, scheduler, and AQM solve different problems: storage absorbs bursts, scheduling chooses who is served, and AQM controls persistent queue pressure.
- Token buckets, policing, shaping, DiffServ, and admission control enforce administrative traffic policy; none creates physical capacity.

Chapter 7 moves above transport to naming, application protocols, caching, and content delivery. The mechanisms in this chapter explain why an application can resolve the right server and establish a reliable connection yet still experience poor completion time when congestion, queue policy, or resource sharing is wrong.